# Option C-extrapolation: target-local time-series projection

**Date:** 2026-04-18.

Current issue: KDE + any selection-based fix can't predict h/m outliers because the cohort doesn't contain enough similar training movies. But at T-3d, the target's own observed arrivals contain information we're not using — we can fit a model to them and project forward.

**Three extrapolation methods:**
1. **`const_rate`** — flat projection: `predicted = (observed_count / observed_window_days) × phase_1_days`.
2. **`last_day_rate`** — use rate in the final 1 day of observed window × phase_1_days. Better for decaying movies.
3. **`exp_decay_fit`** — fit `rate(t) = a·exp(-k·(t - first_review))` to daily observed counts; integrate over phase_1.

**Evaluation set:** 5 h/m movies (the representative-of-deployment subset). Also report cohort-wide for context.

**Caveat from Jake:** `forbidden_fruits_2026` and `they_will_kill_you` have mostly day-level timestamps in their bet windows (scraper wasn't live during their early lifecycle). Extrapolation on them is noisier because cumulative counts are coarsely quantized.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    snapshot_state, passes_skip_rules_for_snap,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0

CACHE = CACHE_DIR / 'option_c_extrapolation.pkl'
print('Ready.')

## Extrapolation methods

In [ ]:
def observed_arrivals(slug, snap_time, target_close):
    """Return list of (dbc, cumulative_count) for arrivals before snap_time, sorted by ts."""
    sub = reviews[
        (reviews['movie_slug'] == slug)
        & (reviews['estimated_timestamp'] < snap_time)
    ].sort_values('estimated_timestamp').copy()
    sub['dbc'] = (target_close - sub['estimated_timestamp']).dt.total_seconds() / 86400
    return sub['dbc'].values


def extrap_const_rate(dbcs, first_review_dbc, snap_dbc, midnight_utc_dbc):
    """Flat rate projection: observed_count / observed_window_days × phase_1_days."""
    n = len(dbcs)
    obs_window_days = first_review_dbc - snap_dbc
    if obs_window_days <= 0:
        return 0.0
    rate = n / obs_window_days
    phase_1_days = snap_dbc - midnight_utc_dbc
    return rate * phase_1_days


def extrap_last_day_rate(dbcs, first_review_dbc, snap_dbc, midnight_utc_dbc):
    """Use rate in final 1 day of observed window × phase_1_days."""
    last_day_mask = (dbcs >= snap_dbc) & (dbcs < snap_dbc + 1.0)
    n_last_day = last_day_mask.sum()
    phase_1_days = snap_dbc - midnight_utc_dbc
    return n_last_day * phase_1_days


def extrap_exp_decay(dbcs, first_review_dbc, snap_dbc, midnight_utc_dbc):
    """Fit rate(t) = a * exp(-k * t_since_first_review), integrate over phase_1.

    Returns (prediction, fit_params_dict).
    Uses daily bucketed counts to handle both h/m and day-level targets.
    """
    if len(dbcs) < 3:
        # Too few points to fit; fall back to constant
        return extrap_const_rate(dbcs, first_review_dbc, snap_dbc, midnight_utc_dbc), None

    # Bucket into days (since first review)
    # t_since_first = first_review_dbc - dbc (so t=0 at first review, grows away from close)
    t_since_first = first_review_dbc - dbcs
    t_max_obs = first_review_dbc - snap_dbc
    # Daily buckets: [0, 1, 2, ..., ceil(t_max_obs)]
    bin_edges = np.arange(0, np.ceil(t_max_obs) + 1)
    counts, _ = np.histogram(t_since_first, bins=bin_edges)
    # Bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    # Drop zero-count bins (log-fit doesn't handle them)
    nonzero = counts > 0
    if nonzero.sum() < 3:
        return extrap_const_rate(dbcs, first_review_dbc, snap_dbc, midnight_utc_dbc), None

    xs = bin_centers[nonzero]
    ys = counts[nonzero].astype(float)

    # Fit log(rate) = log(a) - k*t via linear regression on log(counts)
    # (Assumes exponential decay)
    try:
        slope, intercept = np.polyfit(xs, np.log(ys), 1)
        k = -slope
        a = np.exp(intercept)
    except (ValueError, np.linalg.LinAlgError):
        return extrap_const_rate(dbcs, first_review_dbc, snap_dbc, midnight_utc_dbc), None

    # Integrate rate from snap_dbc to midnight_utc_dbc (i.e., t from t_max_obs to first_review_dbc - midnight_utc_dbc)
    t_start = t_max_obs  # t at snap (days since first review)
    t_end = first_review_dbc - midnight_utc_dbc  # t at midnight UTC of close day
    if abs(k) < 1e-9:
        pred = a * (t_end - t_start)
    else:
        pred = a / k * (np.exp(-k * t_start) - np.exp(-k * t_end))
    return float(max(0, pred)), {'a': float(a), 'k': float(k)}


# Sanity check
for slug in ['the_drama', 'they_will_kill_you', 'forbidden_fruits_2026']:
    close_ts = close_date_map[slug]
    snap_time = close_ts - pd.Timedelta(days=SNAP)
    state = snapshot_state(slug, snap_time)
    if state is None:
        continue
    midnight_utc_dbc = (close_ts - close_ts.floor('D')).total_seconds() / 86400
    dbcs = observed_arrivals(slug, snap_time, close_ts)
    print(f'\n{slug}: obs={len(dbcs)} reviews, first_review_dbc={state["first_review_dbc"]:.2f}, midnight_utc_dbc={midnight_utc_dbc:.3f}')
    p1 = extrap_const_rate(dbcs, state['first_review_dbc'], SNAP, midnight_utc_dbc)
    p2 = extrap_last_day_rate(dbcs, state['first_review_dbc'], SNAP, midnight_utc_dbc)
    p3, fit = extrap_exp_decay(dbcs, state['first_review_dbc'], SNAP, midnight_utc_dbc)
    print(f'  const_rate:    {p1:.2f}')
    print(f'  last_day_rate: {p2:.2f}')
    print(f'  exp_decay_fit: {p3:.2f}  fit={fit}')

## Full cohort comparison at T-3d

For each target, compute KDE prediction (weighted n=20), all three extrapolations, and actual_phase1.

In [ ]:
def run_sweep(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    skipped = {'no_scores': 0, 'kde_failed': 0}
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - SNAP
        if target_window_days <= 0:
            continue
        target_critics = state['observed_critics']

        # Ground truth
        mr = reviews[reviews['movie_slug'] == target].copy()
        mr['dbc'] = (target_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((mr['dbc'] > midnight_utc_dbc) & (mr['dbc'] <= SNAP)).sum())

        # Weighted KDE (control)
        scores = combined_score_with_scores(
            target, target_gap, target_critics, target_window_days,
            k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
        )
        if len(scores) < 5:
            skipped['no_scores'] += 1
            continue

        try:
            profiles = build_weighted_critic_profiles(reviews, close_date_map, scores, verbose=False)
            if len(profiles.df) == 0:
                skipped['kde_failed'] += 1
                continue
            model = build_weighted_kde_lambda_model(
                profiles, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
            )
            pred_kde = predict_window_custom(
                model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
                observed_critics=target_critics,
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
        except Exception as e:
            skipped['kde_failed'] += 1
            continue

        # Extrapolations
        dbcs = observed_arrivals(target, snap_time, target_close)
        pred_const = extrap_const_rate(dbcs, state['first_review_dbc'], SNAP, midnight_utc_dbc)
        pred_last_day = extrap_last_day_rate(dbcs, state['first_review_dbc'], SNAP, midnight_utc_dbc)
        pred_exp, fit = extrap_exp_decay(dbcs, state['first_review_dbc'], SNAP, midnight_utc_dbc)

        rows.append({
            'target': target,
            'target_gap': target_gap,
            'first_review_dbc': state['first_review_dbc'],
            'observed_count': state['observed_count'],
            'pred_kde': float(pred_kde),
            'pred_const': float(pred_const),
            'pred_last_day': float(pred_last_day),
            'pred_exp': float(pred_exp),
            'actual_phase1': actual_p1,
            'exp_decay_k': fit['k'] if fit else np.nan,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    if skipped['no_scores'] or skipped['kde_failed']:
        print(f'Skipped: {skipped}')
    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

results = run_sweep()
print(f'\nn={len(results)}')


## H/m subset — primary evaluation

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm = results[results['target'].isin(HM)].copy()

cols = ['target', 'first_review_dbc', 'observed_count', 'actual_phase1',
        'pred_kde', 'pred_const', 'pred_last_day', 'pred_exp', 'exp_decay_k']
print('H/m predictions:')
print(hm[cols].to_string(index=False, float_format='%.2f'))
print()

print('H/m MAE by method:')
for method in ['kde', 'const', 'last_day', 'exp']:
    errs = hm[f'pred_{method}'] - hm['actual_phase1']
    mae = errs.abs().mean()
    me = errs.mean()
    print(f'  {method:10s}  MAE={mae:6.2f}  mean_err={me:+6.2f}')

## Full cohort comparison (for context)

In [ ]:
print('Full cohort MAE by method (n={}):'.format(len(results)))
for method in ['kde', 'const', 'last_day', 'exp']:
    errs = results[f'pred_{method}'] - results['actual_phase1']
    mae = errs.abs().mean()
    me = errs.mean()
    print(f'  {method:10s}  MAE={mae:6.2f}  mean_err={me:+6.2f}')
print()

# Stratified by actual_phase1 quartile
results['q_actual'] = pd.qcut(results['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
for q in ['Q1','Q2','Q3','Q4']:
    sub = results[results['q_actual'] == q]
    if not len(sub):
        continue
    lo, hi = int(sub['actual_phase1'].min()), int(sub['actual_phase1'].max())
    print(f'{q} (actual [{lo}, {hi}]) n={len(sub)}:')
    for method in ['kde', 'const', 'last_day', 'exp']:
        errs = sub[f'pred_{method}'] - sub['actual_phase1']
        mae = errs.abs().mean()
        me = errs.mean()
        print(f'  {method:10s}  MAE={mae:6.2f}  mean_err={me:+6.2f}')
    print()

## Blend exploration

If extrapolation helps at high n_obs but hurts at low n_obs, a blended predictor `w × KDE + (1-w) × extrap` with w dependent on n_obs might be the ship form.

In [ ]:
# Simple test: weighted avg with w_kde = min(1, 40 / n_obs) — rely on extrap when obs large
results['w_kde'] = np.minimum(1.0, 40.0 / results['observed_count'].clip(lower=1))

for blend_with in ['const', 'last_day', 'exp']:
    results[f'pred_blend_{blend_with}'] = (
        results['w_kde'] * results['pred_kde'] + (1 - results['w_kde']) * results[f'pred_{blend_with}']
    )

print('Blended predictions (w_kde = min(1, 40/n_obs)):')
for method in ['kde', 'blend_const', 'blend_last_day', 'blend_exp']:
    errs = results[f'pred_{method}'] - results['actual_phase1']
    mae = errs.abs().mean()
    me = errs.mean()
    print(f'  {method:15s}  cohort MAE={mae:6.2f}  mean_err={me:+6.2f}')

print()
print('H/m subset blends:')
hm = results[results['target'].isin(HM)].copy()
for method in ['kde', 'blend_const', 'blend_last_day', 'blend_exp']:
    errs = hm[f'pred_{method}'] - hm['actual_phase1']
    mae = errs.abs().mean()
    me = errs.mean()
    print(f'  {method:15s}  h/m MAE={mae:6.2f}  mean_err={me:+6.2f}')